# MergeSort — Divide, Ordena y Vence

**Curso:** Programación Avanzada  
**Sesión:** 17  
**Kernel:** xeus-cling (C++17)

---

Hoy vamos a construir uno de los algoritmos de ordenamiento más elegantes y usados en la industria. En el camino vamos a ver cómo el **paso por referencia** no es solo una técnica de funciones sueltas, sino el mecanismo que hace posible que funciones distintas colaboren sobre los mismos datos sin copiarlos.

- **Divide y vencerás**: parte el problema en mitades hasta que sea trivial
- **Caso base**: una fila de una persona siempre está ordenada
- **Merge**: unir dos filas ya ordenadas en una sola fila ordenada
- **Diseño de API**: cómo la firma de una función comunica su contrato

La idea central: **si sabes ordenar mitades, sabes ordenar el todo**.

---

## Requisitos previos

Antes de comenzar, ejecuta esta celda para cargar las librerías:

In [ ]:
#include <iostream>
#include <vector>
using namespace std;

---

## Parte 1 — La fila del salón

Imagina que ocho estudiantes están parados en fila en el centro del salón, cada uno sosteniendo un número:

```
Posición inicial (desordenados):
┌────┬────┬────┬────┬────┬────┬────┬────┐
│ 20 │ 31 │ 83 │ 92 │ 14 │ 42 │ 51 │ 63 │
└────┴────┴────┴────┴────┴────┴────┴────┘
  [0]  [1]  [2]  [3]  [4]  [5]  [6]  [7]
```

**¿Cómo los ordenamos?** Una idea: dividir la fila por la mitad, ordenar cada mitad por separado, y luego unirlas.

```
Dividimos por la mitad (mid = 3):
┌────┬────┬────┬────┐       ┌────┬────┬────┬────┐
│ 20 │ 31 │ 83 │ 92 │       │ 14 │ 42 │ 51 │ 63 │
└────┴────┴────┴────┘       └────┴────┴────┴────┘
  [0]  [1]  [2]  [3]          [4]  [5]  [6]  [7]

¿Y cómo ordenamos cada mitad? ¡Dividiendo de nuevo!
┌────┬────┐  ┌────┬────┐   ┌────┬────┐  ┌────┬────┐
│ 20 │ 31 │  │ 83 │ 92 │   │ 14 │ 42 │  │ 51 │ 63 │
└────┴────┘  └────┴────┘   └────┴────┘  └────┴────┘

¿Y esas? ¡Dividimos otra vez!
┌────┐ ┌────┐ ┌────┐ ┌────┐  ┌────┐ ┌────┐ ┌────┐ ┌────┐
│ 20 │ │ 31 │ │ 83 │ │ 92 │  │ 14 │ │ 42 │ │ 51 │ │ 63 │
└────┘ └────┘ └────┘ └────┘  └────┘ └────┘ └────┘ └────┘
```

**Una persona sola ya está ordenada — no hace falta hacer nada.**  
Ese es el **caso base**: cuando la fila tiene un solo elemento, está lista.

A partir de ahí, empezamos a *unir* filas ordenadas de menor a mayor hasta reconstruir la fila completa, ya ordenada.

---

## Parte 2 — Divide y vencerás

### 2.1 El árbol de llamadas

MergeSort es un algoritmo **divide y vencerás**: parte el problema a la mitad en cada nivel hasta llegar a casos triviales, y luego construye la solución de abajo hacia arriba.

```
mergesort(arr, 0, 7)
├── mergesort(arr, 0, 3)
│   ├── mergesort(arr, 0, 1)
│   │   ├── mergesort(arr, 0, 0)  ← caso base (1 elemento)
│   │   └── mergesort(arr, 1, 1)  ← caso base (1 elemento)
│   │   └── merge(arr, 0, 0, 1)   ← une [20] y [31]
│   └── mergesort(arr, 2, 3)
│       ├── mergesort(arr, 2, 2)  ← caso base
│       └── mergesort(arr, 3, 3)  ← caso base
│       └── merge(arr, 2, 1, 3)   ← une [83] y [92]
│   └── merge(arr, 0, 1, 3)       ← une [20,31] y [83,92]
└── mergesort(arr, 4, 7)
    └── ... (mismo proceso)
    └── merge(arr, 4, 5, 7)       ← une mitad derecha

merge(arr, 0, 3, 7)               ← une ambas mitades → resultado final
```

El árbol tiene **log₂(8) = 3 niveles**. Cada nivel procesa los 8 elementos una vez. Total: **O(n log n)** operaciones — mucho mejor que el O(n²) de Bubble Sort.

### 2.2 El punto medio

Para dividir un subarreglo `[left, right]` a la mitad, calculamos:

```cpp
int mid = (left + right) / 2;
```

La mitad izquierda cubre `[left, mid]`; la derecha, `[mid+1, right]`.

In [ ]:
{
    // Para el arreglo completo [0, 7]:
    int left = 0, right = 7;
    int mid = (left + right) / 2;

    cout << "Subarreglo: [" << left << ", " << right << "]" << endl;
    cout << "mid = " << mid << endl;
    cout << "Mitad izquierda: [" << left << ", " << mid   << "]" << endl;
    cout << "Mitad derecha:   [" << mid+1 << ", " << right << "]" << endl;
}

Subarreglo: [0, 7]
mid = 3
Mitad izquierda: [0, 3]
Mitad derecha:   [4, 7]


---

## Parte 3 — El caso base

La recursión se detiene cuando `left >= right`: un subarreglo de cero o un elemento ya está ordenado por definición.

In [ ]:
{
    // Simulamos el caso base: left >= right
    auto esCasoBase = [](int left, int right) {
        return left >= right;
    };

    cout << "left=0, right=7: caso base? " << (esCasoBase(0,7) ? "sí":"no") << endl;
    cout << "left=3, right=3: caso base? " << (esCasoBase(3,3) ? "sí":"no") << endl;
    cout << "left=5, right=2: caso base? " << (esCasoBase(5,2) ? "sí":"no") << endl;
}

left=0, right=7: caso base? no
left=3, right=3: caso base? sí
left=5, right=2: caso base? sí


**¿Por qué `left >= right` y no solo `left == right`?**  
Por seguridad: si por algún error de cálculo `left` supera a `right`, la función igual termina correctamente en lugar de entrar en recursión infinita.

---

## Parte 4 — Unir dos filas ordenadas: `merge`

### 4.1 La idea

Tienes dos filas ya ordenadas. Quieres unirlas en una sola fila ordenada.  
Estrategia: compara el primero de cada fila y toma el menor, repite hasta vaciar ambas.

```
Fila izquierda: [20, 31, 83, 92]
Fila derecha:   [14, 42, 51, 63]

Paso 1: ¿20 ≤ 14? No  → toma 14  │ temp: [14]            │ der: [42,51,63]
Paso 2: ¿20 ≤ 42? Sí  → toma 20  │ temp: [14,20]         │ izq: [31,83,92]
Paso 3: ¿31 ≤ 42? Sí  → toma 31  │ temp: [14,20,31]      │ izq: [83,92]
Paso 4: ¿83 ≤ 42? No  → toma 42  │ temp: [14,20,31,42]   │ der: [51,63]
Paso 5: ¿83 ≤ 51? No  → toma 51  │ temp: [14,20,31,42,51]│ der: [63]
Paso 6: ¿83 ≤ 63? No  → toma 63  │ temp: [14,20,31,42,51,63]│ der: vacía
Resto:  copia izq restante [83,92]│ temp: [14,20,31,42,51,63,83,92] ✓
```

### 4.2 ¿Por qué necesitamos un buffer temporal?

No podemos ordenar el arreglo *in place* durante el merge sin sobrescribir datos que todavía necesitamos leer. La solución es copiar los resultados a un arreglo temporal y luego devolverlos al arreglo original.

---

## Parte 5 — Herramienta nueva: `std::vector`

Para el buffer temporal de `merge` necesitamos un arreglo cuyo tamaño se decide en tiempo de ejecución (depende de cuántos elementos tiene el subarreglo). Hasta ahora todos nuestros arreglos tenían tamaño fijo en tiempo de compilación.

**`std::vector<T>`** es un arreglo dinámico de la STL: su tamaño se especifica al crearlo y puede crecer después si se necesita. Para MergeSort solo usaremos la creación con tamaño fijo.

```cpp
vector<int> temp(n);    // crea un vector de n enteros, todos inicializados en 0
temp[i] = valor;        // acceso igual que un arreglo
```

> Para saber más: [cppreference — std::vector](https://en.cppreference.com/w/cpp/container/vector)

In [ ]:
{
    int n = 5;
    vector<int> temp(n);    // arreglo dinámico de n enteros

    for (int i = 0; i < n; i++)
        temp[i] = (i + 1) * 10;

    cout << "temp: ";
    for (int i = 0; i < n; i++)
        cout << temp[i] << " ";
    cout << endl;
    cout << "tamaño: " << temp.size() << endl;
}

temp: 10 20 30 40 50 
tamaño: 5


---

## Parte 6 — Paso por referencia y diseño de la API

### 6.1 La firma de `mergesort`

```cpp
void mergesort(int* arr, int left, int right)
```

Cada parámetro tiene un rol preciso:

| Parámetro | Tipo | Rol |
|---|---|---|
| `arr` | `int*` | **Referencia** al arreglo original — permite modificarlo in-place |
| `left` | `int` | Índice de inicio del subarreglo actual |
| `right` | `int` | Índice de fin del subarreglo actual |

Observa que **no hay copia del arreglo en ningún momento**. Todas las llamadas recursivas operan sobre el mismo bloque de memoria, delimitado por `left` y `right`. Esto es lo que hace al algoritmo eficiente en memoria.

### 6.2 La firma de `merge`

```cpp
void merge(int* arr, int left, int mid, int right)
```

`merge` recibe cuatro parámetros que describen completamente el trabajo a hacer:
- El arreglo (por referencia vía puntero)
- Los límites de la mitad izquierda: `[left, mid]`
- Los límites de la mitad derecha: `[mid+1, right]`

La función sabe exactamente qué sección procesar sin necesitar información global.

### 6.3 Diseño de APIs: comunicar el contrato

Una firma bien diseñada comunica el **contrato** de la función:

```
mergesort(arr, 0, 7)   →  "ordena arr entre el índice 0 y el 7"
merge(arr, 0, 3, 7)    →  "une las mitades [0,3] y [4,7] de arr"
```

Cuando una función recibe un puntero, está diciendo: *"voy a trabajar directamente sobre tus datos"*. Cuando recibe por valor, dice: *"solo necesito leer esto"*.

> Esta distinción —puntero para modificar, valor para leer— es la base del diseño de APIs en C y C++. En APIs modernas de C++ se expresa con `T&` y `const T&`, pero el principio es el mismo.

---

## Parte 7 — Implementación

### 7.1 Función `merge`

Une dos mitades ya ordenadas del arreglo: `[left, mid]` y `[mid+1, right]`.

**Pasos:**
1. Crea un `vector<int> temp` de tamaño `right - left + 1`
2. Con dos índices `i = left` y `j = mid+1`, compara `arr[i]` con `arr[j]` y copia el menor a `temp`
3. Cuando uno de los dos lados se acabe, copia el resto del otro
4. Copia `temp` de vuelta a `arr[left..right]`

In [ ]:
void merge(int* arr, int left, int mid, int right) {
    // Tu implementación aquí
}

### 7.2 Función `mergesort`

Divide el subarreglo a la mitad, ordena cada mitad recursivamente y luego las une.

**Pasos:**
1. Caso base: si `left >= right`, regresa inmediatamente
2. Calcula `mid = (left + right) / 2`
3. Llama a `mergesort` para la mitad izquierda
4. Llama a `mergesort` para la mitad derecha
5. Llama a `merge` para unir las dos mitades

In [ ]:
void mergesort(int* arr, int left, int right) {
    // Tu implementación aquí
}

### 7.3 Prueba

Una vez que implementes las dos funciones, ejecuta esta celda para verificar:

In [ ]:
{
    int arr[] = {20, 31, 83, 92, 14, 42, 51, 63};

    cout << "Arreglo sin ordenar: " << endl;
    for (int i = 0; i < 8; i++)
        cout << arr[i] << " ";
    cout << endl;

    mergesort(arr, 0, 7);

    cout << "Arreglo ordenado:" << endl;
    for (int i = 0; i < 8; i++)
        cout << arr[i] << " ";
    cout << endl;
}

---

## Resumen

| Concepto | Descripción |
|---|---|
| **Divide y vencerás** | Partir el problema a la mitad recursivamente hasta llegar al caso base |
| **Caso base** | `left >= right`: un subarreglo de 0 ó 1 elementos ya está ordenado |
| **`merge`** | Une dos mitades ordenadas en una sola pasada lineal |
| **`std::vector<T>`** | Arreglo dinámico; `vector<int> v(n)` crea un vector de `n` enteros |
| **`int* arr` en la firma** | Paso por referencia: la función trabaja directamente sobre el arreglo original |
| **Complejidad** | O(n log n) en todos los casos — a diferencia de Bubble Sort O(n²) |
| **MergeSort estable** | Preserva el orden relativo de elementos iguales — útil en bases de datos |

---
*Programación Avanzada — Universidad Panamericana*